# FreshRetailNet EDA

This is a reproducible, decision-oriented EDA for FreshRetailNet-50K and FreshRetailNet-LT.

## Decision and grain

One row represents one **store-product-day**. `hours_sale` and `hours_stock_status` contain hourly sequences for that day. The intended decisions are daily demand forecasting, latent-demand recovery during stockouts, replenishment, and waste/service-level trade-offs.

The notebook covers: schema/provenance, integrity, coverage, demand behavior, missingness, seasonality, store-product heterogeneity, leakage/availability, simple baselines, decision-cost assumptions, and report export.

In [ ]:
%pip install -q -U pandas pyarrow matplotlib

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

SEED = 42
MAX_PLOT_ROWS = 250_000
MAX_BASELINE_ENTITIES = 10_000
SEASONAL_LAG = 7
UNDERFORECAST_COST = 3.0
OVERFORECAST_COST = 1.0

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORT_DIR = PROJECT_ROOT / 'reports' / 'eda'
DATASETS = {
    'FreshRetailNet-50K': {
        'root': PROJECT_ROOT / 'data' / 'FreshRetailNet-50K' / 'data',
        'url': 'https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K',
    },
    'FreshRetailNet-LT': {
        'root': PROJECT_ROOT / 'data' / 'FreshRetailNet-LT' / 'data',
        'url': 'https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-LT',
    },
}

files = {(name, split): spec['root'] / f'{split}.parquet'
         for name, spec in DATASETS.items()
         for split in ('train', 'eval')}
missing_files = [str(path) for path in files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('Run download_fresh_retailnet.ipynb first. Missing: ' + ', '.join(missing_files))

ENTITY_COLS = ['city_id', 'store_id', 'product_id']
KEY_COLS = ENTITY_COLS + ['date']
TARGET_COL = 'sale_amount'
NUMERIC_COLS = [
    'sale_amount', 'stock_hour6_22_cnt', 'discount', 'holiday_flag',
    'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level',
]
print(f'Project: {PROJECT_ROOT}')
print(f'Report output: {REPORT_DIR}')

## Schema and provenance

The descriptions below come from each dataset's Hugging Face README. Encoded IDs are identifiers, not measurements; `sale_amount` is globally normalized by the dataset publisher.

In [ ]:
DESCRIPTIONS = {
    'city_id': 'Encoded city identifier',
    'store_id': 'Encoded store identifier',
    'management_group_id': 'Encoded management group identifier',
    'first_category_id': 'Encoded first-level category identifier',
    'second_category_id': 'Encoded second-level category identifier',
    'third_category_id': 'Encoded third-level category identifier',
    'product_id': 'Encoded product/SKU identifier',
    'dt': 'Calendar date as YYYY-MM-DD',
    'sale_amount': 'Daily sales amount after global normalization',
    'hours_sale': '24 hourly sales values after global normalization',
    'stock_hour6_22_cnt': 'Count of out-of-stock hours from 06:00 through 22:00',
    'hours_stock_status': 'Hourly out-of-stock status sequence',
    'discount': 'Discount rate; 1.0 means no discount',
    'holiday_flag': 'Holiday indicator',
    'activity_flag': 'Activity/promotion indicator',
    'precpt': 'Total precipitation',
    'avg_temperature': 'Average temperature',
    'avg_humidity': 'Average humidity',
    'avg_wind_level': 'Average wind force',
}

schema_rows = []
for (dataset, split), path in files.items():
    for field in pq.ParquetFile(path).schema_arrow:
        schema_rows.append({
            'dataset': dataset,
            'split': split,
            'column': field.name,
            'parquet_type': str(field.type),
            'description': DESCRIPTIONS.get(field.name, 'See dataset README'),
            'source': DATASETS[dataset]['url'],
        })
data_dictionary = pd.DataFrame(schema_rows).drop_duplicates(['dataset', 'column'])
display(data_dictionary)

In [ ]:
# Load only the columns needed for this EDA. The hourly sequence columns remain on disk
# and are intentionally excluded from the daily aggregate analysis below.
READ_COLS = ['dt'] + [c for c in ENTITY_COLS + NUMERIC_COLS if c not in {'date'}]
parts = []
for (dataset, split), path in files.items():
    part = pd.read_parquet(path, columns=READ_COLS, engine='pyarrow')
    part = part.rename(columns={'dt': 'date'})
    part['date_raw'] = part['date'].astype('string')
    part['date'] = pd.to_datetime(part['date'], errors='coerce')
    part['dataset'] = dataset
    part['split'] = split
    parts.append(part)
df = pd.concat(parts, ignore_index=True)
del parts, part
print(f'{len(df):,} rows loaded across {df.dataset.nunique()} datasets and {df.split.nunique()} splits')
display(df.head())

## Structural integrity

The primary time key is `(city_id, store_id, product_id, date)`. We check duplicate records, date parsing/formatting, store-to-city consistency, and within-entity ordering/gaps.

In [ ]:
integrity_rows = []
for (dataset, split), g in df.groupby(['dataset', 'split'], sort=False):
    sorted_g = g.sort_values(KEY_COLS)
    gaps = sorted_g.groupby(ENTITY_COLS, sort=False)['date'].diff().dt.days
    gap_by_entity = sorted_g.assign(gap_days=gaps).groupby(ENTITY_COLS)['gap_days'].max()
    ordered_by_entity = g.groupby(ENTITY_COLS, sort=False)['date'].apply(lambda s: s.is_monotonic_increasing)
    store_city_counts = g.groupby('store_id')['city_id'].nunique()
    integrity_rows.append({
        'dataset': dataset,
        'split': split,
        'rows': len(g),
        'unique_keys': g[KEY_COLS].drop_duplicates().shape[0],
        'duplicate_key_rows': int(g.duplicated(KEY_COLS).sum()),
        'invalid_dates': int(g['date'].isna().sum()),
        'non_iso_date_strings': int((~g['date_raw'].str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False)).sum()),
        'stores_in_multiple_cities': int((store_city_counts > 1).sum()),
        'entities_out_of_order': int((~ordered_by_entity).sum()),
        'entities_with_gaps': int((gap_by_entity > 1).sum()),
        'max_gap_days': int(gaps.max()) if gaps.notna().any() else 0,
    })
integrity_report = pd.DataFrame(integrity_rows)
display(integrity_report)
assert integrity_report['invalid_dates'].sum() == 0, 'Invalid dates found'

## Time coverage

Coverage is reported at split level and by day. A complete calendar for an entity is not assumed: gaps and changing entity counts are evidence to carry into model validation.

In [ ]:
coverage_rows = []
for (dataset, split), g in df.groupby(['dataset', 'split'], sort=False):
    span = (g['date'].max() - g['date'].min()).days + 1
    observed_days = g['date'].nunique()
    coverage_rows.append({
        'dataset': dataset,
        'split': split,
        'start_date': g['date'].min().date(),
        'end_date': g['date'].max().date(),
        'calendar_span_days': span,
        'observed_days': observed_days,
        'missing_calendar_days': span - observed_days,
        'unique_store_product_entities': g[ENTITY_COLS].drop_duplicates().shape[0],
        'unique_stores': g['store_id'].nunique(),
        'unique_products': g['product_id'].nunique(),
    })
coverage_report = pd.DataFrame(coverage_rows)
daily_rows = (df.groupby(['dataset', 'split', 'date']).size()
              .reset_index(name='rows'))
daily_entities = (df[['dataset', 'split', 'date'] + ENTITY_COLS].drop_duplicates()
                  .groupby(['dataset', 'split', 'date']).size()
                  .reset_index(name='entities'))
daily_coverage = daily_rows.merge(daily_entities, on=['dataset', 'split', 'date'])
display(coverage_report)
display(daily_coverage.head())

In [ ]:
fig_coverage, axes = plt.subplots(len(DATASETS), 1, figsize=(12, 4 * len(DATASETS)), squeeze=False)
for ax, (dataset, _) in zip(axes[:, 0], DATASETS.items()):
    for split, g in daily_coverage[daily_coverage['dataset'].eq(dataset)].groupby('split'):
        ax.plot(g['date'], g['entities'], label=split)
    ax.set_title(f'{dataset}: entities observed by day')
    ax.set_ylabel('unique store-product entities')
    ax.legend()
    ax.grid(alpha=0.2)
fig_coverage.tight_layout()
plt.show()

## Target behavior and stockout signal

`sale_amount` is normalized daily sales. Zeros are not automatically stockouts; the operational stockout flag is `stock_hour6_22_cnt > 0`.

In [ ]:
target_rows = []
for (dataset, split), g in df.groupby(['dataset', 'split'], sort=False):
    y = g[TARGET_COL].dropna()
    p99 = y.quantile(0.99) if len(y) else np.nan
    target_rows.append({
        'dataset': dataset, 'split': split, 'rows': len(g),
        'mean': y.mean(), 'std': y.std(), 'p50': y.quantile(0.50),
        'p90': y.quantile(0.90), 'p99': p99, 'p999': y.quantile(0.999),
        'max': y.max(), 'zero_pct': (y == 0).mean() * 100,
        'negative_count': int((y < 0).sum()),
        'spike_pct_above_p99': (y > p99).mean() * 100 if len(y) else np.nan,
        'stockout_day_pct': (g['stock_hour6_22_cnt'].fillna(0) > 0).mean() * 100,
    })
target_summary = pd.DataFrame(target_rows)
display(target_summary)

plot_df = df.sample(min(MAX_PLOT_ROWS, len(df)), random_state=SEED)
fig_target, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log1p(plot_df[TARGET_COL].clip(lower=0)), bins=60, color='steelblue')
axes[0].set_title('log1p daily sales')
axes[0].set_xlabel('log1p(sale_amount)')
axes[1].boxplot([plot_df.loc[plot_df['stock_hour6_22_cnt'].fillna(0).eq(0), TARGET_COL].dropna(),
                    plot_df.loc[plot_df['stock_hour6_22_cnt'].fillna(0).gt(0), TARGET_COL].dropna()],
                   labels=['no stockout', 'stockout'])
axes[1].set_title('sales by stockout-day flag')
axes[1].set_ylabel('normalized sale_amount')
fig_target.tight_layout()
plt.show()

## Missingness map

Missingness is summarized globally, by date, and for stores/products. A missing operational field may be informative rather than random, so the maps are retained as candidate features for the data-quality review.

In [ ]:
missing_rows = []
for (dataset, split), g in df.groupby(['dataset', 'split'], sort=False):
    for column in READ_COLS:
        actual = 'date_raw' if column == 'dt' else column
        missing_rows.append({
            'dataset': dataset, 'split': split, 'column': column,
            'missing_count': int(g[actual].isna().sum()),
            'missing_pct': g[actual].isna().mean() * 100,
        })
missingness_map = pd.DataFrame(missing_rows)
display(missingness_map.sort_values('missing_pct', ascending=False))

missing_columns = [c for c in NUMERIC_COLS if df[c].isna().any()]
if missing_columns:
    missing_by_day = (df.assign(any_missing=df[missing_columns].isna().any(axis=1))
                      .groupby(['dataset', 'split', 'date'])['any_missing'].mean()
                      .reset_index(name='any_missing_rate'))
    display(missing_by_day.sort_values('any_missing_rate', ascending=False).head(20))
    for dimension in ['store_id', 'product_id']:
        by_entity = (df.groupby(['dataset', 'split', dimension])[missing_columns]
                     .apply(lambda x: x.isna().mean().mean(), include_groups=False)
                     .reset_index(name='mean_missing_rate'))
        display(by_entity.sort_values('mean_missing_rate', ascending=False).head(20))
else:
    print('No missing values found in the loaded daily analysis columns.')

## Seasonality and dynamics

The profiles below compare weekday effects, holiday/activity effects, stockouts, and time trends. They are descriptive only; any feature used by a model must still pass the availability audit.

In [ ]:
df['day_of_week'] = df['date'].dt.dayofweek
df['day_name'] = df['date'].dt.day_name().str[:3]
df['month'] = df['date'].dt.to_period('M').astype(str)
df['stockout_day'] = df['stock_hour6_22_cnt'].fillna(0).gt(0)

dow_profile = (df.groupby(['dataset', 'split', 'day_of_week', 'day_name'])
                .agg(mean_sales=(TARGET_COL, 'mean'), median_sales=(TARGET_COL, 'median'),
                     stockout_rate=('stockout_day', 'mean'), rows=(TARGET_COL, 'size'))
                .reset_index().sort_values('day_of_week'))
event_profile = (df.groupby(['dataset', 'split', 'holiday_flag', 'activity_flag'])
                   .agg(mean_sales=(TARGET_COL, 'mean'), median_sales=(TARGET_COL, 'median'),
                        stockout_rate=('stockout_day', 'mean'), rows=(TARGET_COL, 'size'))
                   .reset_index())
daily_profile = (df.groupby(['dataset', 'split', 'date'])
                   .agg(total_sales=(TARGET_COL, 'sum'), mean_sales=(TARGET_COL, 'mean'),
                        stockout_rate=('stockout_day', 'mean'), entities=(TARGET_COL, 'size'))
                   .reset_index())
display(dow_profile)
display(event_profile)

fig_seasonal, axes = plt.subplots(1, 2, figsize=(14, 4))
for (dataset, split), g in dow_profile.groupby(['dataset', 'split']):
    axes[0].plot(g['day_name'], g['mean_sales'], marker='o', label=f'{dataset} / {split}')
axes[0].set_title('Mean daily sales by weekday')
axes[0].set_ylabel('mean sale_amount')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)

for (dataset, split), g in daily_profile.groupby(['dataset', 'split']):
    axes[1].plot(g['date'], g['total_sales'].rolling(7, min_periods=1).mean(), label=f'{dataset} / {split}')
axes[1].set_title('7-day rolling total sales')
axes[1].set_ylabel('normalized sales')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)
fig_seasonal.tight_layout()
plt.show()

## Cross-sectional heterogeneity

Entities are segmented using observed coverage, zero-sales rate, and coefficient of variation. This is a screening segmentation for model routing, not a final business taxonomy.

In [ ]:
entity_summary = (df.groupby(['dataset', 'split'] + ENTITY_COLS)
                    .agg(rows=(TARGET_COL, 'size'), first_date=('date', 'min'), last_date=('date', 'max'),
                         total_sales=(TARGET_COL, 'sum'), mean_sales=(TARGET_COL, 'mean'),
                         std_sales=(TARGET_COL, 'std'), zero_rate=(TARGET_COL, lambda x: (x == 0).mean()),
                         stockout_rate=('stockout_day', 'mean'))
                    .reset_index())
entity_summary['cv'] = entity_summary['std_sales'] / entity_summary['mean_sales'].replace(0, np.nan)
entity_summary['coverage_days'] = (entity_summary['last_date'] - entity_summary['first_date']).dt.days + 1

def segment(row):
    if row['coverage_days'] and row['rows'] / row['coverage_days'] < 0.8:
        return 'sparse_coverage'
    if row['zero_rate'] >= 0.5:
        return 'intermittent'
    if pd.notna(row['cv']) and row['cv'] >= 1.0:
        return 'volatile'
    return 'stable'

entity_summary['segment'] = entity_summary.apply(segment, axis=1)
segment_counts = (entity_summary.groupby(['dataset', 'split', 'segment'])
                  .size().reset_index(name='entities'))
display(segment_counts)

store_sales = (df.groupby(['dataset', 'split', 'store_id'])[TARGET_COL].sum()
               .reset_index(name='sales').sort_values(['dataset', 'split', 'sales'], ascending=[True, True, False]))
product_sales = (df.groupby(['dataset', 'split', 'product_id'])[TARGET_COL].sum()
                 .reset_index(name='sales').sort_values(['dataset', 'split', 'sales'], ascending=[True, True, False]))
for label, table in [('top stores', store_sales), ('top products', product_sales)]:
    print(label)
    display(table.groupby(['dataset', 'split']).head(10))

fig_heterogeneity, axes = plt.subplots(1, 2, figsize=(14, 4))
for (dataset, split), g in entity_summary.groupby(['dataset', 'split']):
    axes[0].hist(np.log1p(g['mean_sales'].clip(lower=0)), bins=50, alpha=0.45, label=f'{dataset} / {split}')
    axes[1].hist(g['zero_rate'], bins=30, alpha=0.45, label=f'{dataset} / {split}')
axes[0].set_title('Entity mean-sales distribution')
axes[0].set_xlabel('log1p(mean daily sales)')
axes[1].set_title('Entity zero-sales rate')
axes[1].set_xlabel('zero rate')
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig_heterogeneity.tight_layout()
plt.show()

## Leakage and feature availability audit

The forecast origin is assumed to be the start of the day being forecast. A feature marked conditional is usable only if its value is known or forecasted before that origin. In particular, same-day stockout counts/status and same-day realized sales are not automatically valid predictors.

In [ ]:
leakage_audit = pd.DataFrame([
    ['city_id, store_id, product_id', 'identifier', 'yes', 'none', 'Safe entity keys; do not treat as numeric magnitude'],
    ['date / day-of-week / month', 'calendar', 'yes', 'none', 'Known at forecast origin'],
    ['holiday_flag', 'calendar plan', 'conditional', 'possible', 'Use only when the holiday calendar is known in advance'],
    ['activity_flag', 'promotion plan', 'conditional', 'possible', 'Use only if activity is scheduled/known before origin'],
    ['discount', 'price/promotion', 'conditional', 'possible', 'Use planned discount, not a realized post-decision value'],
    ['precpt, avg_temperature, avg_humidity, avg_wind_level', 'weather', 'conditional', 'possible', 'Use historical values only for lagged features or an available weather forecast'],
    ['sale_amount', 'target', 'no for same-day forecast', 'yes', 'Use only as lagged history or the label'],
    ['hours_sale', 'target sequence', 'no for same-day forecast', 'yes', 'Contains realized same-day sales'],
    ['stock_hour6_22_cnt', 'stockout operational signal', 'conditional', 'yes', 'Same-day value is post-origin; lag it or define an intraday forecast origin'],
    ['hours_stock_status', 'stock availability sequence', 'conditional', 'yes', 'Same-day future hours leak availability; use only observed history at origin'],
], columns=['field', 'role', 'available_at_day_start', 'leakage_risk', 'recommendation'])
display(leakage_audit)

## Baseline signal

A one-step rolling-origin backtest on a deterministic sample of training entities compares naive, 7-day seasonal-naive, and 7-day moving-average forecasts. The cutoff is chronological, so no future training rows are used for a prediction.

In [ ]:
train = df[df['split'].eq('train')].copy()
group_cols = ['dataset'] + ENTITY_COLS
eligible = (train.groupby(group_cols).size()
            .loc[lambda s: s >= SEASONAL_LAG + 3])
selected = eligible.sample(min(MAX_BASELINE_ENTITIES, len(eligible)), random_state=SEED).reset_index()[group_cols]
baseline = train.merge(selected, on=group_cols, how='inner').sort_values(group_cols + ['date']).copy()
grouped_y = baseline.groupby(group_cols, sort=False)[TARGET_COL]
baseline['naive'] = grouped_y.shift(1)
baseline['seasonal_naive_7d'] = grouped_y.shift(SEASONAL_LAG)
baseline['moving_average_7d'] = grouped_y.transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
backtest_start = train['date'].quantile(0.80)
test = baseline[baseline['date'] >= backtest_start].copy()

def score(prediction):
    valid = test[[TARGET_COL, prediction]].dropna()
    error = valid[prediction] - valid[TARGET_COL]
    return {
        'forecast': prediction,
        'n': len(valid),
        'mae': error.abs().mean(),
        'rmse': np.sqrt((error ** 2).mean()),
        'underforecast_rate': (error < 0).mean(),
        'weighted_error_cost': (UNDERFORECAST_COST * (-error).clip(lower=0) + OVERFORECAST_COST * error.clip(lower=0)).mean(),
    }

baseline_scores = pd.DataFrame([score(c) for c in ['naive', 'seasonal_naive_7d', 'moving_average_7d']])
display(baseline_scores.sort_values('mae'))
assert not baseline_scores.empty and baseline_scores['n'].gt(0).all(), 'Baseline produced no valid rows'

## Decision relevance

These are explicit starting assumptions, not measured business costs. Underforecasting is assigned a higher cost because it can create stockouts/lost service; overforecasting can create waste or markdown exposure. Replace these values with business estimates before model selection.

In [ ]:
decision_assumptions = pd.DataFrame([
    ['underforecast', UNDERFORECAST_COST, 'Lost sales, stockout exposure, lower service level'],
    ['overforecast', OVERFORECAST_COST, 'Waste, markdowns, holding/handling cost'],
    ['forecast origin', 'day start', 'Same-day future sales and stockout states unavailable'],
    ['service-level target', 'to be supplied', 'Set with operations before optimizing a model'],
], columns=['error_type', 'assumption', 'decision_consequence'])
display(decision_assumptions)
print('Replace the placeholder assumptions with observed lost-sales, waste, and service-level costs before production decisions.')

## Reproducible report export

All key tables and figures are written to `reports/eda/`. The seed, parameters, source locations, and row counts are saved in a manifest so another run can be compared with this one.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
tables = {
    'data_dictionary': data_dictionary,
    'integrity_report': integrity_report,
    'coverage_report': coverage_report,
    'daily_coverage': daily_coverage,
    'target_summary': target_summary,
    'missingness_map': missingness_map,
    'dow_profile': dow_profile,
    'event_profile': event_profile,
    'daily_profile': daily_profile,
    'entity_summary': entity_summary,
    'segment_counts': segment_counts,
    'leakage_audit': leakage_audit,
    'baseline_scores': baseline_scores,
    'decision_assumptions': decision_assumptions,
}
for name, table in tables.items():
    table.to_csv(REPORT_DIR / f'{name}.csv', index=False)

for name, figure in [('coverage', fig_coverage), ('target', fig_target), ('seasonal', fig_seasonal), ('heterogeneity', fig_heterogeneity)]:
    figure.savefig(REPORT_DIR / f'{name}.png', dpi=140, bbox_inches='tight')

manifest = {
    'seed': SEED,
    'max_plot_rows': MAX_PLOT_ROWS,
    'max_baseline_entities': MAX_BASELINE_ENTITIES,
    'seasonal_lag': SEASONAL_LAG,
    'underforecast_cost': UNDERFORECAST_COST,
    'overforecast_cost': OVERFORECAST_COST,
    'source_urls': {name: spec['url'] for name, spec in DATASETS.items()},
    'file_rows': {f'{dataset}/{split}': int(len(df[(df['dataset'] == dataset) & (df['split'] == split)]))
                  for dataset, split in files},
}
(REPORT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print(f'Wrote {len(tables)} tables, 4 figures, and a manifest to {REPORT_DIR}')